# 02 — UMAP & Clustering

This notebook covers:
1. Loading the preprocessed AnnData object
2. Building a neighbor graph
3. Computing UMAP embedding
4. Clustering with the Leiden algorithm
5. Visualizing clusters and marker genes

**Run 01_preprocessing.ipynb first.**

In [ ]:
import sys
sys.path.insert(0, '..')

import scanpy as sc
from src import visualization as viz
from src import utils

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white')

print('Setup complete.')

## 1. Load Preprocessed Data

In [ ]:
adata = utils.load_h5ad('pbmc3k_preprocessed.h5ad')  # saved by notebook 01
utils.summarize(adata, 'Preprocessed')

## 2. Neighbor Graph

Before UMAP, we build a k-nearest-neighbor graph in PCA space.
- `n_neighbors` controls local vs global structure (10-50 is typical)
- `n_pcs` should match the 'elbow' you identified in the PCA scree plot

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=20)
print('Neighbor graph built.')

## 3. UMAP

UMAP projects the high-dimensional PCA space into 2D for visualization.
Cells that are similar in gene expression land near each other.

In [ ]:
sc.tl.umap(adata)
print('UMAP computed.')

## 4. Leiden Clustering

`resolution` controls granularity:
- Lower (e.g. 0.3) → fewer, broader clusters
- Higher (e.g. 1.0) → more, finer clusters

Start with 0.5 and adjust based on biology.

In [ ]:
sc.tl.leiden(adata, resolution=0.5)
print(f"Clusters found: {adata.obs['leiden'].nunique()}")
print(adata.obs['leiden'].value_counts())

## 5. Visualize Clusters

In [ ]:
# Color by cluster label
viz.plot_umap(adata, color=['leiden'], title='UMAP — Leiden clusters', save_name='umap_leiden.png')

In [ ]:
# Color by QC metrics to check data quality
viz.plot_umap(
    adata,
    color=['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    title='UMAP — QC metrics',
    save_name='umap_qc.png'
)

## 6. Find Marker Genes

Rank genes that are differentially expressed in each cluster compared to all others.  
This helps identify **what cell type each cluster is**.

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon')

# Show top 5 marker genes per cluster
sc.pl.rank_genes_groups(adata, n_genes=5, sharey=False)

In [ ]:
# Dot plot: expression of known marker genes across clusters
# (Swap these for brain markers like NEUROD2, SOX2, GFAP when using real data)
marker_genes = ['IL7R', 'CD14', 'LYZ', 'MS4A1', 'CD8A', 'GNLY', 'NKG7']

sc.pl.dotplot(adata, marker_genes, groupby='leiden', standard_scale='var')

## 7. Save Final Object

In [ ]:
utils.save_processed(adata, 'pbmc3k_clustered.h5ad')
print('Done. Ready for trajectory analysis in the next notebook.')